In [ ]:
import pyarrow.parquet as pq
import polars as pl
import numpy as np
from tqdm.auto import tqdm
import gc

In [ ]:
TEST_PATH = '/kaggle/input/datasets/b22dckh072/file05/test_interactions.parquet'
FINAL_TOP100_PATH = '/kaggle/input/datasets/b22dckh072/file05/top100_final_recommendations.parquet' 
ITEM_MAPPING_PATH = '/kaggle/input/datasets/b22dckh072/file05/item_mapping.parquet'

In [ ]:
print("Đang nạp tập Test và tạo danh sách đối chiếu...")
df_test = pl.read_parquet(TEST_PATH)
truth_df = df_test.group_by('mapped_user_id').agg(pl.col('mapped_item_id').alias('true_items'))
n_valid = truth_df.height

print(f"Số lượng người dùng hợp lệ trong tập Test: {n_valid:,}")
print("Đang tính toán chỉ số bằng luồng dữ liệu (Streaming Evaluation)...")

FINAL_TOP100_PATH = '/kaggle/input/datasets/b22dckh072/feature-engineering/top100_final_recommendations.parquet'
pf = pq.ParquetFile(FINAL_TOP100_PATH)

In [ ]:
# --- BƯỚC 1: LỌC TẬP TEST (CHỈ LẤY NGƯỜI DÙNG LẺ) ---
print("Đang chuẩn bị tập Test (Người dùng lẻ)...")
# Giả sử bạn đã nạp truth_df từ trước, đây là lệnh lọc:
truth_df = truth_df.filter((pl.col('mapped_user_id') % 2) != 0)

# Tính tổng số người dùng hợp lệ để chia trung bình ở cuối
n_valid = truth_df.select('mapped_user_id').n_unique()
print(f"Số lượng người dùng trong tập Test: {n_valid:,}")

# --- BƯỚC 2: ĐÁNH GIÁ THEO CHUNK ---
# Lấy 5 triệu dòng mỗi nhịp để đạt tốc độ tối đa mà không tốn RAM
reader = pf.iter_batches(batch_size=5000000, columns=['mapped_user_id', 'mapped_item_id'])

hr_sum = 0
ndcg_sum = 0.0
rc100_sum = 0  # Biến mới để lưu tổng Recall@100
buffer_df = pl.DataFrame()

for batch in tqdm(reader, desc="Evaluating Chunks"):
    chunk = pl.from_arrow(batch)
    
    # Nối phần cắt dở từ chunk trước
    if buffer_df.height > 0:
        chunk = pl.concat([buffer_df, chunk])
        
    last_user = chunk.get_column('mapped_user_id')[-1]
    
    # Tách những user đã có đầy đủ danh sách ứng viên
    completed = chunk.filter(pl.col('mapped_user_id') != last_user)
    buffer_df = chunk.filter(pl.col('mapped_user_id') == last_user)
    
    if completed.height > 0:
        # Đánh số thứ tự dựa trên vị trí hiện tại
        completed = completed.with_columns(
            pl.int_range(1, pl.len() + 1).over('mapped_user_id').alias('rank')
        )
        
        # Lấy Top 100 để đánh giá RC@100 trước
        top100 = completed.filter(pl.col('rank') <= 100)
        
        # Kết nối với danh sách mua thật trong tập Test
        eval_df_100 = top100.join(truth_df, on='mapped_user_id', how='inner')
        
        # Kiểm tra trúng đích (Hit) trong Top 100
        hits_100 = eval_df_100.filter(pl.col('mapped_item_id').is_in(pl.col('true_items')))
        
        if hits_100.height > 0:
            # Cộng dồn số User có Hit vào Recall@100
            rc100_sum += hits_100.select('mapped_user_id').n_unique()
            
            # Lọc tiếp Top 10 từ danh sách Hit của Top 100 để tính HR@10 và NDCG@10
            hits_10 = hits_100.filter(pl.col('rank') <= 10)
            
            if hits_10.height > 0:
                hr_sum += hits_10.select('mapped_user_id').n_unique()
                
                # Tính NDCG: 1 / log2(rank + 1) và cộng dồn
                ndcg = hits_10.with_columns((1.0 / np.log2(pl.col('rank') + 1)).alias('ndcg_val'))
                ndcg_sum += ndcg.select(pl.col('ndcg_val').sum()).item()
                
        del top100, eval_df_100, hits_100
        
    del chunk, completed
    gc.collect()

# --- BƯỚC 3: XỬ LÝ ĐOẠN BUFFER CUỐI CÙNG ---
if buffer_df.height > 0:
    buffer_df = buffer_df.with_columns(
        pl.int_range(1, pl.len() + 1).over('mapped_user_id').alias('rank')
    )
    top100 = buffer_df.filter(pl.col('rank') <= 100)
    eval_df_100 = top100.join(truth_df, on='mapped_user_id', how='inner')
    hits_100 = eval_df_100.filter(pl.col('mapped_item_id').is_in(pl.col('true_items')))
    
    if hits_100.height > 0:
        rc100_sum += hits_100.select('mapped_user_id').n_unique()
        
        hits_10 = hits_100.filter(pl.col('rank') <= 10)
        if hits_10.height > 0:
            hr_sum += hits_10.select('mapped_user_id').n_unique()
            ndcg = hits_10.with_columns((1.0 / np.log2(pl.col('rank') + 1)).alias('ndcg_val'))
            ndcg_sum += ndcg.select(pl.col('ndcg_val').sum()).item()

# --- BƯỚC 4: TÍNH TOÁN VÀ IN KẾT QUẢ ---
HR10 = hr_sum / n_valid
NDCG10 = ndcg_sum / n_valid
RC100 = rc100_sum / n_valid

print("=======================================")
print(f"KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH XGBOOST")
print(f"Tập người dùng hợp lệ (Lẻ): {n_valid:,}")
print(f"Recall @ 100 (RC@100):      {RC100:.4f}")
print(f"Hit Rate @ 10 (HR@10):      {HR10:.4f}")
print(f"NDCG @ 10:                  {NDCG10:.4f}")
print("=======================================")

In [ ]:
print("\n--- BƯỚC 5: DỊCH NGƯỢC ITEM MAPPING ---")
print("Đang thiết lập luồng dịch ID (Lazy Scan)...")

# 1. Quét trực tiếp file Mapping định dạng Parquet (Không nạp thẳng vào RAM)
lf_mapping = pl.scan_parquet(ITEM_MAPPING_PATH)

# 2. Quét file Top 100 ở chế độ Lazy
lf_final = pl.scan_parquet(FINAL_TOP100_PATH)

print("Đang xử lý luồng cắt Top 10 và Join dữ liệu...")

# Thiết lập toàn bộ chuỗi thao tác (Chưa chạy thật)
lf_readable_recs = (
    lf_final
    .with_columns(pl.int_range(1, pl.len() + 1).over('mapped_user_id').alias('rank'))
    .filter(pl.col('rank') <= 10)
    .join(lf_mapping, on='mapped_item_id', how='left')
    .select(['mapped_user_id', 'parent_asin', 'score', 'rank'])
)

print("Đang thực thi và xuất file (Streaming Mode)...")
FINAL_CSV_PATH = '/kaggle/working/final_top10_recommendations.csv'

# 3. COLLECT VỚI STREAMING=TRUE: Polars sẽ tự động chia nhỏ data để Join và Ghi
final_readable_recs = lf_readable_recs.collect(streaming=True)

final_readable_recs.write_csv(FINAL_CSV_PATH)

print(f"Đã xuất danh sách gợi ý thực tế tại: {FINAL_CSV_PATH}")
print("\nXem thử 5 gợi ý đầu tiên:")
print(final_readable_recs.head(5))

# Dọn dẹp cuối cùng
del final_readable_recs
gc.collect()